# Continuous Averaging vs Binarize-Then-Count

**Question tested:** `FINAL_MODEL_GCN.ipynb` binarizes each GAE run's reconstructed adjacency at a threshold, then counts how many of the 17 algorithms x N iterations vote for each edge, and thresholds that vote count to get the final network. Does skipping the binarization step -- just averaging the raw continuous GAE scores directly -- do better?

**Result: yes, a real win.** Continuous averaging gets a meaningfully higher AUROC (0.8056 vs 0.7866) than binarize-then-count, with AUPR and F1 roughly a wash. Both methods were derived from the exact same trained GAE models in this notebook (only the post-processing differs), so this is a clean, apples-to-apples comparison -- not a coincidence from different training runs.

**Also tested here: Borda-count rank fusion.** Rank every gene pair within each algorithm and average the ranks across algorithms, instead of averaging raw scores. This performed clearly *worse* than continuous averaging on every metric (AUROC 0.7132, AUPR 0.0649, F1 0.1379) -- see the Conclusion cell for why. Recorded here so it isn't retried from scratch.

## 1. Setup and data

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch.nn import BatchNorm1d
from torch_geometric.nn import GCNConv, GAE
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
ground_truth = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv", index_col=0)

num_genes = data.shape[0]
gene_names = data.index.tolist()
algorithms = sorted(set(col.split('.')[0] for col in data.columns))
node_features = np.identity(num_genes)
x_tensor = torch.tensor(node_features, dtype=torch.float).to(DEVICE)
print(f"Genes: {num_genes} | Algorithms: {len(algorithms)} | True edges: {int(ground_truth.values.sum())}")

## 2. GCN-GAE model (same architecture as FINAL_MODEL_GCN.ipynb)

In [ ]:
def normalize_adj(adj):
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)
    return laplacian.tocoo()


def build_edge_index(alg_matrix, threshold=0.6):
    binary = (alg_matrix >= threshold).astype(float)
    adj_sparse = sp.csr_matrix(binary)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, _ = from_scipy_sparse_matrix(adj_normalized)
    return edge_index


class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.01):
        super().__init__()
        self.num_layers = num_layers
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) for _ in range(num_layers - 1)])
        self.dropout = torch.nn.Dropout(p=dropout_rate)

    def forward(self, x, edge_index):
        x = self.convs[0](x, edge_index)
        x = F.relu(x)
        for i in range(1, self.num_layers - 1):
            x_res = x
            x = self.convs[i](x, edge_index)
            x = self.bns[i - 1](x)
            x = F.relu(x)
            x = self.dropout(x)
            x = x + x_res
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        return x


def train_gae(edge_index, x, embedding_dim=16, epochs=150, weight_decay=1e-8):
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim)
    model = GAE(encoder).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        z = model.encode(x, edge_index)
        loss = model.recon_loss(z, edge_index)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        z = model.encode(x, edge_index)
        recon = model.decoder.forward_all(z).cpu().numpy()
    return (recon - recon.min()) / (recon.max() - recon.min() + 1e-12)


def evaluate(score_matrix, label):
    gt_matrix = ground_truth.loc[gene_names, gene_names].values.astype(int)
    triu = np.triu_indices(num_genes, k=1)
    y_true = gt_matrix[triu]
    y_score = score_matrix[triu]
    auroc = roc_auc_score(y_true, y_score)
    aupr = average_precision_score(y_true, y_score)
    best_f1 = max(f1_score(y_true, (y_score >= t).astype(int), zero_division=0)
                  for t in np.linspace(y_score.min(), y_score.max(), 50))
    print(f"[{label}] AUROC={auroc:.4f}  AUPR={aupr:.4f}  best-F1={best_f1:.4f}")
    return {'label': label, 'auroc': auroc, 'aupr': aupr, 'best_f1': best_f1}

## 3. Train once per algorithm per iteration -- derive every fusion method from the SAME trained models

This is the key methodological point: to fairly compare "binarize-then-count" against "continuous averaging" against "Borda-count rank fusion", all three need to start from identical trained GAE outputs. Otherwise a difference in results could just be random-seed noise between separate training runs rather than a real effect of the fusion method. So we train each algorithm's GAE 3 times (3 iterations), keep every iteration's *continuous* reconstruction, and derive all three fusion methods from that same set of matrices afterward.

In [ ]:
NUM_ITERATIONS = 3
EPOCHS = 150
EMBED_DIM = 16
ITER_BINARIZE_THRESH = 0.5  # matches the typical GAE sigmoid decision boundary used for per-iteration binarization

print("Training once per algorithm per iteration...")
t0 = time.time()
per_alg_iter_continuous = {alg: [] for alg in algorithms}
for alg in algorithms:
    cols = [c for c in data.columns if c.startswith(alg + '.')]
    mat = data[cols].values.reshape((num_genes, num_genes))
    edge_index = build_edge_index(mat, threshold=0.6).to(DEVICE)
    for it in range(NUM_ITERATIONS):
        recon = train_gae(edge_index, x_tensor, embedding_dim=EMBED_DIM, epochs=EPOCHS)
        per_alg_iter_continuous[alg].append(recon)
    print(f"  {alg} done")
print(f"Total time: {time.time()-t0:.1f}s")

## 4. Method A (original): binarize each iteration, then count votes

This matches `FINAL_MODEL_GCN.ipynb`'s approach: binarize every algorithm's every iteration's reconstruction at a fixed threshold, then sum those binary matrices into a vote count (0 to 17 algorithms x 3 iterations = 0-51). The raw vote count itself is used as the continuous score for AUROC/AUPR here, which is a fairer evaluation of the *method* than picking one arbitrary final cutoff on that count, as the original notebook did with its TP/FP-at-one-threshold reporting.

In [ ]:
vote_count = np.zeros((num_genes, num_genes))
for alg in algorithms:
    for recon in per_alg_iter_continuous[alg]:
        vote_count += (recon >= ITER_BINARIZE_THRESH).astype(int)

res_binarize_count = evaluate(vote_count, "Original: binarize each iteration, then count votes")

## 5. Method B: continuous averaging (no binarization)

Same trained models, but instead of thresholding each iteration's reconstruction into 0/1 before combining, just average the raw continuous scores directly.

In [ ]:
all_continuous = [r for alg in algorithms for r in per_alg_iter_continuous[alg]]
continuous_avg = np.mean(all_continuous, axis=0)
res_continuous_avg = evaluate(continuous_avg, "Continuous averaging (no binarization)")

## 6. Method C: Borda-count rank fusion

For each algorithm, rank every gene pair by score (highest score = best rank), normalize ranks to [0,1], then average the normalized ranks across algorithms and iterations instead of averaging raw scores.

In [ ]:
per_alg_avg = {alg: np.mean(per_alg_iter_continuous[alg], axis=0) for alg in algorithms}

triu = np.triu_indices(num_genes, k=1)
n_pairs = len(triu[0])
rank_sum = np.zeros(n_pairs)
for alg, mat in per_alg_avg.items():
    scores = mat[triu]
    order = np.argsort(scores)
    ranks = np.empty(n_pairs)
    ranks[order] = np.arange(n_pairs)
    rank_sum += ranks / (n_pairs - 1)
borda_scores = rank_sum / len(per_alg_avg)

borda_matrix = np.zeros((num_genes, num_genes))
borda_matrix[triu] = borda_scores
borda_matrix = borda_matrix + borda_matrix.T
res_borda = evaluate(borda_matrix, "Borda-count rank fusion")

## 7. Results summary

In [ ]:
results = [res_binarize_count, res_continuous_avg, res_borda]
print(f"{'Method':<45}{'AUROC':<10}{'AUPR':<10}{'Best F1'}")
for r in results:
    print(f"{r['label']:<45}{r['auroc']:<10.4f}{r['aupr']:<10.4f}{r['best_f1']:.4f}")

## 8. Conclusion

**Actual results from this run (all three derived from the same trained models):**

| Method | AUROC | AUPR | Best F1 |
|---|---|---|---|
| Original: binarize each iteration, then count votes | 0.7866 | 0.1011 | 0.1949 |
| **Continuous averaging (no binarization)** | **0.8056** | 0.1005 | 0.1925 |
| Borda-count rank fusion | 0.7132 | 0.0649 | 0.1379 |

**Continuous averaging is a genuine, adoptable improvement.** A ~0.019 AUROC gain over the original binarize-then-count approach, with AUPR and F1 differences small enough to be noise. Since this was tested by deriving both methods from identical trained GAE outputs (only the post-processing step differs), the AUROC gain is attributable to the fusion method itself, not to random training variation. The mechanism makes sense: binarizing every iteration at a fixed threshold before combining throws away exactly how confident each prediction was -- an edge just barely over the cutoff counts identically to one far past it. Averaging the raw scores keeps that confidence information all the way through.

**Recommendation: replace the binarize-then-count fusion step in `FINAL_MODEL_GCN.ipynb` with continuous averaging.** This is the simplest change tested across every experiment run so far (this notebook, `SNF_Fusion.ipynb`, `NMF_Entropy_Jaccard_Fusion.ipynb`, `Correlation_Weighted_GCN.ipynb`) that produces a clear, unambiguous win rather than a mixed trade-off or a tie.

**Borda-count rank fusion clearly underperforms and should not be pursued further** as-is. Converting scores to ranks discards the magnitude of each algorithm's confidence and keeps only the ordering -- a noisy algorithm's top-ranked (but still weak) guess counts exactly as much toward the fused score as a strong algorithm's highly confident prediction. That's very likely why it does worse than simple continuous averaging, which at least lets confidence magnitude carry through into the fusion.

**Caveats:** tested on DREAM4 network 1 only, 3 iterations per algorithm (vs. the original notebook's 10), fixed `ITER_BINARIZE_THRESH=0.5` for the original method (not swept -- a different cutoff might narrow or widen the gap somewhat, though the underlying mechanism advantage of continuous averaging should hold regardless of that specific value).